<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">07 Miscelleaneous</h1>
<hr>

## **1. Comparing Processor Architectures**

<div class="alert alert-light">

  Name: (name)

  Matriculation Number: (matriculation number)

</div>

### 1.1 Introduction

<div style="display:flex;align-items:center;justify-content:center;gap:2%;width:100%;margin:0 auto;">
  <img src="../../.admin/img/xPUs.png" alt="xPUs" style="width:80%;height:auto;filter:grayscale(100%);">
</div>


Computers use different processors for different kinds of work. A CPU (central processing unit) runs the operating system and general application code, handles program logic, and coordinates devices. A GPU (graphics processing unit) performs many similar operations in parallel, supporting graphics, image processing, scientific computing, and neural-network training and inference. An NPU (neural processing unit) specializes in neural-network operations, commonly accelerating inference tasks such as image recognition and speech processing.

This notebook compares the CPU and built-in VideoCore VII GPU of a Raspberry Pi 5 with the Hailo-10H NPU on an AI HAT+ 2. The theory section uses matrix-multiplication animations to illustrate sequential execution, parallel outputs, and streamed computation. The Raspberry Pi control console then displays the hardware inventory and live operating conditions during experiments.

Three experiments connect these ideas to measured performance. Matrix multiplication compares five CPU and GPU implementations, including different thread counts and tiled and untiled computation. ResNet-50 classifies the same image using one CPU core, four CPU cores, the GPU, and the NPU, with the three highest-scoring classes shown for each implementation. Qwen2-1.5B-Instruct answers the same three prompts on CPU, GPU, and NPU; the actual responses, response times, and token counts support comparison of speed and answer quality.

The learning objectives are to relate processor architecture to computation, distinguish execution time from data-transfer and runtime overhead, and interpret latency and throughput alongside numerical precision and output quality. Matching tasks make comparisons more informative, but quantization, software implementations, and operating conditions still affect the results. All performance measurements run on the Pi; the notebook host displays the animations, console, and results.


#### 1.1 Theory

The diagram below illustrates how CPUs, GPUs, and NPUs organize computation and memory. CPU cores execute flexible instruction sequences, including branches and vector arithmetic. GPUs distribute parallel work across many execution lanes. For NPUs, a compiler maps supported neural-network operations onto specialized compute and memory resources. These capabilities overlap, and performance depends on software that can use the available hardware. The diagram presents design overviews rather than transistor layouts or core counts. Modern CPUs support vector instructions, GPUs can include matrix units, and NPUs also require control logic and memory. For background, see NVIDIA's SIMT explanation [\[01\]](#ref-01) and Hailo's model deployment flow [\[02\]](#ref-02). The GPU picture is a general abstraction - VideoCore VII does not implement NVIDIA's architecture.

<p style="text-align: center;"><img src="../../.admin/notebooks/cpu_gpu_npu/img/processor-architecture.png" alt="CPU, GPU and NPU compute and memory organization" width="80%" style="width:80%;height:auto;"></p>


In the following, three animations illustrate matrix multiplication on a CPU, GPU and NPU. For two square matrices $A$ and $B$, each with $n$ rows and $n$ columns, $C = A B$ and $C_{ij}=\sum_{k=0}^{n-1}A_{ik}B_{kj}$. One output combines a row of $A$ with a column of $B$. Every processor must compute the same mathematical answer, within the error allowed by its numeric representation. MAC stands for multiply-accumulate: multiply two numbers and add their product to a running sum. One MAC performs the update

$$
s \leftarrow s + a \times b.
$$

For example, with $s=10$, $a=3$, and $b=4$, one MAC changes the running sum to $10+3\times4=22$. Matrix multiplication repeats this update, starting each output's sum at zero. Each output of a 16 × 16 matrix product requires 16 MACs. With 256 outputs, the complete product requires $256\times16=4{,}096$ MACs. More generally, a dense $n\times n$ product requires $n^3$ MACs.

Performance figures commonly count one MAC as two arithmetic operations: one multiplication and one addition. The workload is therefore approximately $2n^3$ operations, or 8,192 operations for the 16 × 16 example. This convention includes adding the first product to zero; counting only the additions needed to sum $n$ products gives $n-1$ additions per output.

##### 1.1.1 CPU: one output at a time

CPU matrix multiplication computes one output at a time. Play advances through all 256 output cells at 70 ms per multiplication step. Completed values remain visible. This is a scalar teaching view; optimized CPU libraries typically compute vectorized tiles.

In [ ]:
%run ../../.admin/notebooks/cpu_gpu_npu/show_processor_animation.py cpu-matrix-16x16


##### 1.1.2 GPU: parallel outputs

GPU matrix multiplication computes outputs in parallel. Sixteen partial sums advance together in each tile of matrix C. Playback continues across all tiles until matrix C is filled. This illustrates parallel tiled multiplication, not the exact execution units or timing of VideoCore VII.

In [ ]:
%run ../../.admin/notebooks/cpu_gpu_npu/show_processor_animation.py gpu-matrix-16x16


##### 1.1.3 NPU: streamed computation

The NPU illustration shows a grid of connected multiply-accumulate units. Each unit keeps one output's partial sum locally, multiplies the two arriving values, adds the product, and passes the input values to its neighbours. This regular movement is called a systolic array. The animation illustrates the principle; it does not describe the internal layout of Hailo-10H.

The arithmetic is the same on all three processors. For the first output tile, start each partial sum at zero and apply

$$
s_{ij}^{(k+1)}=s_{ij}^{(k)}+A_{ik}B_{kj},\qquad s_{ij}^{(0)}=0.
$$

After 16 updates, $s_{ij}^{(16)}=C_{ij}$. In the simplified GPU animation, 16 outputs advance through the same index $k$ together. In the NPU animation, each output has a fixed position in the grid, and input values travel between neighbouring units. An $A$ value travels right and is reused across four output columns; a $B$ value travels downward and is reused across four output rows. The partial sum stays in its unit. This arrangement is called output-stationary dataflow.

The diagonal wave follows directly from these travel times. Number the rows and columns of the first 4 × 4 tile from zero. At animation tick $t$, the unit at $(i,j)$ receives the pair with

$$
k=t-i-j.
$$

It updates its sum only when $0\leq k<16$. Thus, at tick $t=0$, unit $(0,0)$ adds $A_{00}B_{00}$. At tick $t=1$, that unit adds $A_{01}B_{10}$, while unit $(0,1)$ starts with $A_{00}B_{01}$ and unit $(1,0)$ starts with $A_{10}B_{00}$. The same input values are being reused at neighbouring positions. The animation labels these zero-based ticks as ticks 1, 2, and so on. For this illustrative 4 × 4 array, the last unit finishes at $t=15+3+3=21$: 22 ticks including filling and draining the pipeline. Both animations still perform $4\times4\times16=256$ MACs per tile. Their tick counts and playback speeds are not hardware performance measurements.

The potential benefit is local data reuse and a regular hardware schedule, which can reduce data-movement and instruction overhead. Optimized GPUs also reuse data, and some contain specialized matrix units, so this is not an exclusive GPU/NPU distinction. An NPU additionally specializes its supported operations, numeric formats, and compiler scheduling for neural networks. A typical layer is $Y=\phi(XW+b)$, where $W$ contains weights, $b$ is a bias, and $\phi$ is an activation function. The compiler maps these operations to supported hardware and manages their intermediate data [\[02\]](#ref-02).

During playback, the advancing diagonal in matrix C represents the arrival of operands at successive units. Completed values remain visible as the animation continues through all output tiles.

Remark: A TPU (tensor processing unit) is Google's specialized AI accelerator for neural-network training and inference. It illustrates the same broader principle as an NPU: hardware tailored to tensor operations, particularly matrix multiplication. Google's TPUs use systolic arrays of connected multiply-accumulate units, making them a useful reference for the animation [\[08\]](#ref-08). TPU is a specific accelerator family; NPU is a broader category. The Raspberry Pi in this notebook uses a Hailo NPU.


In [ ]:
%run ../../.admin/notebooks/cpu_gpu_npu/show_processor_animation.py npu-matrix-16x16


### 1.2 Raspberry Pi

The Pi 5 has a CPU and VideoCore VII GPU sharing system DRAM. The GPU does not cross the external
PCIe link to access that memory. Shared memory still has bandwidth, synchronization, and cache costs.
The AI HAT+ 2 adds a Hailo-10H and 8 GB of memory on the HAT; host input and output cross PCIe.
Its 40 TOPS rating is at INT4 precision, not a promise of 40 trillion FP32 operations per second.
See the AI HAT specifications [\[03\]](#ref-03).



<div style="display:flex;align-items:center;justify-content:center;gap:2%;width:100%;margin:0 auto;">
  <img src="../../.admin/notebooks/cpu_gpu_npu/img/raspberry-pi-processors.png" alt="Raspberry Pi 5 with CPU and GPU in the same chip, alongside a Pi fitted with the Hailo-10H AI HAT+ 2" style="width:35.942412%;height:auto;">
  <img src="../../.admin/notebooks/cpu_gpu_npu/img/raspberry-pi-hardware.png" alt="Raspberry Pi 5 CPU, GPU, shared memory and PCIe connection to AI HAT+ 2" style="width:62.057588%;height:auto;">
</div>


#### 2.1 Raspberry Pi control console

The console connects using the local SSH key and displays the hardware inventory, temperature, storage, memory, processor load, and fan speed. Readings update about once per second. The fan switch controls cooling, and the paper log records status changes.

The console may take a few seconds to appear.


In [ ]:
%run -i ../../.admin/notebooks/cpu_gpu_npu/check_pi_hardware.py

### 1.3 Experiments

Three experiments run on the Raspberry Pi: matrix multiplication compares five CPU and GPU implementations; ResNet-50 classifies the same image on CPU, GPU, and NPU; and Qwen2 answers three prompts once per processor. The results show execution time, throughput, and the corresponding outputs.

Loading and initial setup are excluded from timing. Comparisons apply within each workload, with precision, software, and output quality considered alongside speed. The following subsections describe each measurement; section 4 collects the available results. The control console in section 2.1 can show activity during execution, although monitoring adds some overhead.


#### 3.1 Matrix multiplication: five Pi implementations

All five implementations compute the same 512 × 512 FP32 matrix multiplication on the Raspberry Pi. One warm-up matrix multiplication precedes one timed trial containing three matrix multiplications. Input hashes must match, and every output is checked against a float64 reference with a relative Frobenius error below `1e-4`.

| Implementation | Execution |
|---|---|
| CPU tiled C++, one thread | One CPU worker computes 16 × 16 output tiles |
| CPU tiled C++, four threads | Four OpenMP workers divide the output tiles across CPU cores |
| CPU NumPy/BLAS, four threads | An optimized library performs the same matrix multiplication |
| GPU tiled Vulkan | GPU workgroups compute 16 × 16 output tiles |
| GPU untiled Vulkan | Each GPU invocation computes one output directly from A and B |

The tiled C++ and tiled Vulkan versions use the same tile size and accumulation order over the inner matrix dimension. Each reuses input tiles and keeps partial sums for an output tile. Their execution differs: C++ processes tile elements in vectorizable loops, while Vulkan distributes them across workgroup invocations with synchronization barriers. C++ is compiled with `-O3`, native CPU instructions, OpenMP, and fused multiply-add contraction enabled. The single-thread implementation retains vectorization. The four-thread implementation verifies its actual worker count. These are comparable implementations of tiled multiplication, rather than identical instruction streams.

BLAS stands for Basic Linear Algebra Subprograms, a library interface for operations such as matrix multiplication. A BLAS thread is a CPU worker used by the library to perform part of a calculation in parallel. NumPy's `A @ B` uses the installed BLAS library. Four threads allow work across the Pi's four CPU cores, but coordination overhead and memory bandwidth can limit the speedup. The worker verifies that BLAS reports four threads.

The untiled shader assigns one output element to each invocation and accumulates its row-by-column dot product directly from the input buffers. It uses no shared-memory tiles or workgroup barriers. Both GPU variants retain 16 × 16 workgroups, the same inputs, and the same submission code, so this comparison changes the shader's data-access and synchronization strategy. More repeated reads may increase memory traffic, while removing shared-memory accesses and barriers may reduce overhead; the measured result determines which works better on this Pi.

Both Vulkan variants select the Broadcom GPU and reject CPU software rendering. Each matrix multiplication is submitted separately and followed by a completion wait. Vulkan supports compute shaders without a graphics window [\[09\]](#ref-09). The input and output buffers remain allocated throughout timing. SSH, compilation, allocation, and numerical checks are outside the timer for all implementations.

The comparison reports latency, matrix multiplications per second, speedup relative to the single-thread C++ baseline, and temperatures. A faster GPU result is not assumed. Temperature and throttling readings, compiler settings, input hashes, and raw trial durations accompany the saved results. Before each implementation, the worker waits up to 10 seconds for the Pi to cool to 65 °C or below; the actual starting temperature is recorded.

The control console remains running while the following cell executes all five implementations. Timestamped progress identifies preparation, cooling, and each implementation in the cell output. The console paper records warning-light ON events and fan events. Monitoring adds some overhead; closing the console before the run remains possible for quieter timing conditions. SSH uses the local key. Parameters remain configurable, and results are stored in `matrix_comparison` and saved under `.admin/notebooks/cpu_gpu_npu/results`. Required tools are described in the [Raspberry Pi setup guide](../../.admin/notebooks/cpu_gpu_npu/raspberry_pi_setup.md).


In [ ]:
from pathlib import Path
import runpy

runner_candidates = [
    Path("../../.admin/notebooks/cpu_gpu_npu/run_matrix_comparison.py"),
    Path(".admin/notebooks/cpu_gpu_npu/run_matrix_comparison.py"),
]
runner_path = next((path for path in runner_candidates if path.is_file()), None)
if runner_path is None:
    raise FileNotFoundError("Keep .admin/notebooks/cpu_gpu_npu in the repository checkout.")
runner = runpy.run_path(str(runner_path))
matrix_comparison = None
cpu_results, gpu_results, gpu_result = [], [], None
matrix_comparison = runner["run_comparison"](
    hostname="raspberrypi.local", username="cas", n=512, repetitions=3, trials=1,
)
cpu_results = [r for r in matrix_comparison["results"] if r["processor"] == "CPU"]
gpu_results = [r for r in matrix_comparison["results"] if r["processor"] == "GPU"]
gpu_result = next((r for r in gpu_results if r["shader_variant"] == "tiled"), None)


#### 3.2 ResNet-50 on CPU, GPU, and NPU

This experiment runs the same ResNet-50 v1 image-classification network on all three Pi processors. A shared photograph is cropped and resized to 224 × 224 RGB pixels, and each implementation returns 1,000 ImageNet class scores. The CPU and GPU use the same FP32 TFLite file from Hailo Model Zoo. The NPU uses the corresponding compiled ResNet-50 HEF installed with `hailo-models` [\[14\]](#ref-14), [\[16\]](#ref-16).

<p align="center">Model input: 224 × 224 pixels<br><img src="../../.admin/notebooks/cpu_gpu_npu/img/resnet50-dog-input.png" alt="Centered square crop resized to 224 by 224 pixels" width="224" style="max-width:100%"></p>

The exact model input is shown above [\[16\]](#ref-16). Inference uses a centered square crop resized to 224 × 224 pixels. The table lists the three highest-scoring classes for each implementation. Scores are model outputs expressed as percentages; quantized NPU scores can differ from the FP32 scores and need not sum to exactly 100%.

CPU execution uses LiteRT with one thread pinned to one CPU core, then four threads allowed across four cores. Both runs use the same model and input. GPU execution uses LiteRT's experimental WebGPU/Vulkan path with FP32 enforced; the worker requires full GPU acceleration and increasing hardware V3D counters [\[05\]](#ref-05), [\[15\]](#ref-15). The NPU runs quantized operations through HailoRT and returns dequantized FP32 scores. The task and input match, but precision and runtime optimizations differ. This compares deployed implementations of one network rather than identical arithmetic across processors.

Each processor performs one warm-up and one timed trial of three sequential inferences, with batch size one. Timing starts with the shared RGB array in host memory and ends with completed class scores in host memory. It includes channel-mean subtraction, input staging, execution, output transfer or conversion, and the output copy. Mean subtraction runs before LiteRT and is embedded in the NPU model. Image decoding, cropping, model loading, and initial buffer allocation are excluded. Average latency is total trial time divided by three; one trial provides no variability estimate.

The worker checks model and image hashes, output shape, finite scores, repeatability, and CPU/GPU numerical agreement. It retains the NPU's score differences and each processor's top predictions. Agreement on one photograph does not establish dataset accuracy. Progress appears in the cell output, and measured results are retained in `neural_comparison` and saved under `.admin/notebooks/cpu_gpu_npu/results`.


In [ ]:
from pathlib import Path
import runpy

runner_candidates = [
    Path("../../.admin/notebooks/cpu_gpu_npu/run_neural_comparison.py"),
    Path(".admin/notebooks/cpu_gpu_npu/run_neural_comparison.py"),
]
runner_path = next((path for path in runner_candidates if path.is_file()), None)
if runner_path is None:
    raise FileNotFoundError("Keep .admin/notebooks/cpu_gpu_npu in the repository checkout.")
runner = runpy.run_path(str(runner_path.resolve()))
neural_comparison = None
neural_comparison = runner["run_comparison"](
    hostname="raspberrypi.local", username="cas", repetitions=3,
)


#### 3.3 Qwen2 language model on the Pi

This experiment runs Qwen2-1.5B-Instruct, a large language model (LLM) with approximately 1.5 billion parameters, on the same Raspberry Pi. Three fixed prompts cover a factual question, simple arithmetic, and a short explanation of the CPU. Each prompt uses an independent conversation, greedy decoding, and a limit of 64 generated tokens. Each prompt runs once on CPU, GPU, and NPU. Answers may finish earlier. The system message and ChatML prompt formatting are shared across runtimes [\[10\]](#ref-10), [\[11\]](#ref-11).

The CPU and GPU use the same Q4_0 GGUF model file through llama.cpp. CPU execution uses four threads. GPU execution explicitly selects the Broadcom Vulkan driver and requests model-layer offload. The NPU uses Hailo's compiled Qwen2 HEF through Hailo-Ollama. Its quantized operations differ from GGUF, so this is a comparison of deployed implementations of the same base model. Matching model names do not imply identical arithmetic or identical answers [\[04\]](#ref-04), [\[12\]](#ref-12).

The cell starts one runtime at a time on the Pi, performs a separate warm-up request, and then measures requests over the Pi's loopback interface. Time to first text measures the delay until the first nonempty streamed response fragment; this approximates time to first token. Total response time includes prompt processing, generation, and local HTTP overhead. Tokens per second divides the runtime's reported output token count by total response time, including prompt processing. This is an application throughput measure. Token counts and complete answers are retained because answer lengths and runtime token-count conventions can differ. Loading and warm-up are excluded from the prompt results, and prompt-cache reuse is disabled or reset between requests.

Temperature and throttling flags accompany the results. The worker waits for the Pi to cool between requests when necessary. A failed startup or incomplete response stream appears with its diagnostic and receives no completed-suite timing. An incorrect warm-up answer is flagged alongside any completed measurements. Each prompt appears in the cell output when it starts. Its response, total response time, and output token count appear as soon as the request completes; the elementary warm-up check does not establish model accuracy. Incorrect answers remain visible rather than being removed from the sample.

The models and runtimes are installed under `~/llm-comparison` and the Hailo system directories on the Pi. The following cell runs the complete experiment; the source and prompt list are stored in `.admin/notebooks/cpu_gpu_npu`. SSH login uses the local SSH key. Results are retained in `llm_results` and saved as `~/llm-comparison/llm-results.json` on the Pi.


In [ ]:
from pathlib import Path
import runpy

runner_candidates = [
    Path("../../.admin/notebooks/cpu_gpu_npu/run_llm_benchmarks.py"),
    Path(".admin/notebooks/cpu_gpu_npu/run_llm_benchmarks.py"),
]
llm_runner_path = next((path for path in runner_candidates if path.is_file()), None)
if llm_runner_path is None:
    raise FileNotFoundError("The LLM benchmark helper was not found in .admin/notebooks/cpu_gpu_npu.")
llm_runner = runpy.run_path(str(llm_runner_path.resolve()))
llm_results = llm_runner["run_benchmarks"](
    hostname="raspberrypi.local", username="cas",
    repetitions=1, gpu_repetitions=1, max_tokens=64, backends=("CPU", "GPU", "NPU"),
)


### 1.4 Compare measurements

The following charts and tables collect the current session's matrix, ResNet-50, and Qwen2 measurements, including the NPU classification and language-generation results. Each workload has its own time scale. CPU, GPU, and NPU use consistent colors, and shorter bars indicate lower latency. Missing experiments are identified and can be run before refreshing the comparison.

Matrix results compare matching FP32 calculations. ResNet-50 compares the same image and network across different execution formats; the NPU uses quantized arithmetic. Qwen2 shows each prompt separately, with time to first text, total response time, token count, and throughput. Responses can differ in length and quality, so their text in section 3.3 remains part of the comparison. These short runs do not establish timing variability or dataset accuracy.


In [ ]:
from pathlib import Path
import runpy

candidates = [
    Path("../../.admin/notebooks/cpu_gpu_npu/compare_measurements.py"),
    Path(".admin/notebooks/cpu_gpu_npu/compare_measurements.py"),
]
helper = next((p for p in candidates if p.is_file()), None)
if helper is None:
    raise FileNotFoundError("The comparison helper is missing from .admin/notebooks/cpu_gpu_npu.")
runpy.run_path(str(helper.resolve()))["show_comparison"](
    matrix=globals().get("matrix_comparison"),
    neural=globals().get("neural_comparison"),
    llm=globals().get("llm_results"),
)


GPU performance depends on how much parallel work is available and how efficiently the software uses the hardware. Small calculations can spend a substantial fraction of their time submitting work and waiting for completion. Tiling can reduce repeated memory access, but also adds coordination between GPU workers. The Pi's integrated GPU shares system memory with the CPU, so memory bandwidth and the chosen implementation also affect performance.

Language generation has an additional constraint: each new token depends on the preceding tokens. A GPU can parallelize calculations within a generation step, but successive tokens still require successive steps. Total response time includes both prompt processing and answer generation; longer answers can take more time even when their token throughput is higher.

Interpret comparisons within the same workload and timing boundary. Lower latency means a completed task takes less time; higher throughput means more work finishes per second. Quantized models use different numerical representations, which can affect both speed and outputs. Predicted classes, scores, and generated answers therefore matter alongside timing. These measurements describe the complete hardware and software configuration, and a single short run does not establish a general ranking of processor architectures.


### 1.5 References

<span id="ref-01">[01]</span> NVIDIA. [Writing SIMT Kernels](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/writing-cuda-kernels.html). CUDA Programming Guide, sec. 2.3. Accessed 30 September 2026.

<span id="ref-02">[02]</span> Hailo. [Hailo Model Zoo](https://github.com/hailo-ai/hailo_model_zoo/blob/master/README.rst). GitHub repository, README, master branch. Accessed 30 September 2026.

<span id="ref-03">[03]</span> Raspberry Pi. [AI HATs](https://www.raspberrypi.com/documentation/accessories/ai-hat-plus.html). Raspberry Pi Documentation. Accessed 30 September 2026.

<span id="ref-04">[04]</span> Raspberry Pi. [AI software](https://www.raspberrypi.com/documentation/computers/ai.html). Raspberry Pi Documentation. Accessed 30 September 2026.

<span id="ref-05">[05]</span> Naush Patuck. [Get started with LiteRT on Raspberry Pi](https://www.raspberrypi.com/news/get-started-with-litert-on-raspberry-pi/). Raspberry Pi. 13 August 2026. Accessed 30 September 2026.

<span id="ref-06">[06]</span> Naush Patuck. [Mastering edge AI on Raspberry Pi with LiteRT and Gemma](https://www.raspberrypi.com/news/mastering-edge-ai-on-raspberry-pi-with-litert-and-gemma/). Raspberry Pi. 11 August 2026. Accessed 30 September 2026.

<span id="ref-07">[07]</span> Hailo. [Hailo-Apps](https://github.com/hailo-ai/hailo-apps). GitHub repository, README. Accessed 30 September 2026.

<span id="ref-08">[08]</span> Google Cloud. [TPU architecture](https://docs.cloud.google.com/tpu/docs/system-architecture-tpu-vm). Google Cloud Documentation, “How a TPU works.” Accessed 30 September 2026.

<span id="ref-09">[09]</span> Khronos Group. [Compute Shader](https://docs.vulkan.org/tutorial/latest/11_Compute_Shader.html). Vulkan Tutorial. Accessed 30 September 2026.

<span id="ref-10">[10]</span> Qwen Team. [Qwen2-1.5B-Instruct-GGUF](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct-GGUF). Model card and quantized model files, revision `c62434db644497c0ee545c690bb66a67eba6eb3f`. Accessed 30 September 2026.

<span id="ref-11">[11]</span> ggml-org. [llama.cpp server](https://github.com/ggml-org/llama.cpp/tree/876c75b1f63d1567532fd4300fc5777549619d63/tools/server). Runtime documentation and source, revision `876c75b1f63d1567532fd4300fc5777549619d63`. Accessed 30 September 2026.

<span id="ref-12">[12]</span> Hailo. [Hailo Model Zoo GenAI: Usage](https://github.com/hailo-ai/hailo_model_zoo_genai/blob/v5.1.1/docs/USAGE.rst). Version 5.1.1, Hailo-Ollama API and model deployment. Accessed 30 September 2026.

<span id="ref-13">[13]</span> Vivienne Sze, Yu-Hsin Chen, Tien-Ju Yang, and Joel Emer. [Efficient Processing of Deep Neural Networks: A Tutorial and Survey](https://arxiv.org/abs/1703.09039). arXiv:1703.09039, 2017, version 2 (13 August 2017). Accessed 1 October 2026.

<span id="ref-14">[14]</span> Hailo. [ResNet-50 v1 configuration](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/networks/resnet_v1_50.yaml). Hailo Model Zoo, source model and normalization settings. Accessed 1 October 2026.

<span id="ref-15">[15]</span> Google. [LiteRT Python API](https://developers.google.com/edge/litert/next/python). CompiledModel, accelerator selection, and tensor buffers. Accessed 1 October 2026.

<span id="ref-16">[16]</span> PyTorch. [Example dog photograph](https://github.com/pytorch/hub/blob/master/images/dog.jpg) and [ImageNet class labels](https://github.com/pytorch/hub/blob/master/imagenet_classes.txt). PyTorch Hub repository. Accessed 1 October 2026.


---
### 1.6 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**For matrix multiplication, which GPU implementation has the lower latency: tiled or untiled?**

Answer: 


**For ResNet-50, which CPU configuration is faster: one core or four cores?**

Answer: 


**For ResNet-50, which processor has the lowest latency: CPU, GPU, or NPU?**

Answer: 


**For the first Qwen2 prompt, which processor finishes fastest: CPU, GPU, or NPU?**

Answer: 


**Do all three processors generate the same number of tokens for the first Qwen2 prompt? Yes or no.**

Answer: 


**Is the GPU faster than the CPU in every experiment? Yes or no.**

Answer: 


**OPTIONAL: Are all tasks in this notebook clear and understandable? Are there any shortcomings, missing parts, or sections that should be improved?**

 (add feedback or suggestions for improvement here)